In [ ]:
!pip -q install sentencepiece datasets
import os, math, time, glob, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR='/content/drive/MyDrive/otto_gpt'
SPM_PREFIX=f'{PROJECT_DIR}/tokenizer_ko'
BASE_CKPT=f'{PROJECT_DIR}/ckpt/otto_gpt.pt'            
INST_CKPT=f'{PROJECT_DIR}/ckpt/otto_gpt_instruct.pt'   
assert os.path.exists(BASE_CKPT), '사전학습 체크포인트가 없습니다. 먼저 otto_gpt_scratch 노트북을 돌리세요.'
print('베이스 체크포인트 OK')

In [ ]:
import sentencepiece as spm
sp=spm.SentencePieceProcessor(model_file=SPM_PREFIX+'.model')
print('어휘 크기:', sp.get_piece_size())

In [ ]:
from datasets import load_dataset
ds=load_dataset('beomi/KoAlpaca-v1.1a', split='train')
INST_BIN=f'{PROJECT_DIR}/data/instruct.bin'
if not os.path.exists(INST_BIN):
    ids_all=[]
    for ex in ds:
        q=(ex.get('instruction') or '').strip(); a=str(ex.get('output') or '').strip()
        if len(q)<2 or len(a)<2: continue
        text=f'### 질문: {q}\n### 답변: {a}'
        ids_all.extend(sp.encode(text)); ids_all.append(sp.eos_id())
    arr=np.array(ids_all, dtype=np.uint16)
    arr.tofile(INST_BIN)
    print('instruct 토큰 수:', len(arr))
else:
    print('이미 존재:', INST_BIN, os.path.getsize(INST_BIN)//2, '토큰')

In [ ]:
@dataclass
class GPTConfig:
    vocab_size:int=16000; block_size:int=512; n_layer:int=10; n_head:int=12
    n_embd:int=624; dropout:float=0.1; bias:bool=False
class CausalSelfAttention(nn.Module):
    def __init__(s,c):
        super().__init__(); s.c_attn=nn.Linear(c.n_embd,3*c.n_embd,bias=c.bias)
        s.c_proj=nn.Linear(c.n_embd,c.n_embd,bias=c.bias); s.resid_dropout=nn.Dropout(c.dropout)
        s.n_head,s.n_embd,s.dropout=c.n_head,c.n_embd,c.dropout
    def forward(s,x):
        B,T,C=x.size(); q,k,v=s.c_attn(x).split(s.n_embd,dim=2)
        k=k.view(B,T,s.n_head,C//s.n_head).transpose(1,2); q=q.view(B,T,s.n_head,C//s.n_head).transpose(1,2)
        v=v.view(B,T,s.n_head,C//s.n_head).transpose(1,2)
        y=F.scaled_dot_product_attention(q,k,v,dropout_p=s.dropout if s.training else 0.0,is_causal=True)
        return s.resid_dropout(s.c_proj(y.transpose(1,2).contiguous().view(B,T,C)))
class MLP(nn.Module):
    def __init__(s,c):
        super().__init__(); s.c_fc=nn.Linear(c.n_embd,4*c.n_embd,bias=c.bias); s.gelu=nn.GELU()
        s.c_proj=nn.Linear(4*c.n_embd,c.n_embd,bias=c.bias); s.dropout=nn.Dropout(c.dropout)
    def forward(s,x): return s.dropout(s.c_proj(s.gelu(s.c_fc(x))))
class Block(nn.Module):
    def __init__(s,c):
        super().__init__(); s.ln_1=nn.LayerNorm(c.n_embd,bias=c.bias); s.attn=CausalSelfAttention(c)
        s.ln_2=nn.LayerNorm(c.n_embd,bias=c.bias); s.mlp=MLP(c)
    def forward(s,x): x=x+s.attn(s.ln_1(x)); x=x+s.mlp(s.ln_2(x)); return x
class GPT(nn.Module):
    def __init__(s,c):
        super().__init__(); s.cfg=c
        s.transformer=nn.ModuleDict(dict(wte=nn.Embedding(c.vocab_size,c.n_embd),
            wpe=nn.Embedding(c.block_size,c.n_embd),drop=nn.Dropout(c.dropout),
            h=nn.ModuleList([Block(c) for _ in range(c.n_layer)]),ln_f=nn.LayerNorm(c.n_embd,bias=c.bias)))
        s.lm_head=nn.Linear(c.n_embd,c.vocab_size,bias=False); s.transformer.wte.weight=s.lm_head.weight
    def forward(s,idx,targets=None):
        B,T=idx.size(); pos=torch.arange(0,T,device=idx.device)
        x=s.transformer.drop(s.transformer.wte(idx)+s.transformer.wpe(pos))
        for b in s.transformer.h: x=b(x)
        x=s.transformer.ln_f(x)
        if targets is not None:
            logits=s.lm_head(x); return logits, F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1),ignore_index=-1)
        return s.lm_head(x[:,[-1],:]), None
print('GPT 정의 완료')

In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'
dtype='bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ckpt=torch.load(BASE_CKPT, map_location=device)
config=GPTConfig(**ckpt['config'])
model=GPT(config).to(device); model.load_state_dict(ckpt['model'])
print('베이스 모델 로드 완료. 사전학습 step:', ckpt['iter_num'])
data=np.memmap(INST_BIN, dtype=np.uint16, mode='r')
batch_size=24; block_size=config.block_size
def get_batch():
    ix=torch.randint(len(data)-block_size,(batch_size,))
    x=torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y=torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

In [ ]:
max_iters=4000; lr=1e-4; warmup=100
opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
scaler=torch.amp.GradScaler(enabled=(dtype=='float16'))
def get_lr(it):
    if it<warmup: return lr*(it+1)/warmup
    r=(it-warmup)/(max_iters-warmup); return 1e-5+0.5*(1+math.cos(math.pi*r))*(lr-1e-5)
model.train(); t0=time.time()
for it in range(max_iters):
    for g in opt.param_groups: g['lr']=get_lr(it)
    X,Y=get_batch()
    with torch.amp.autocast(device_type='cuda',dtype=getattr(torch,dtype)):
        _,loss=model(X,Y)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(opt); scaler.update()
    if it%100==0: print(f'iter {it:4d} | loss {loss.item():.4f} | {time.time()-t0:.0f}s')
torch.save({'model':model.state_dict(),'config':config.__dict__,'iter_num':it+1}, INST_CKPT)
print('튜닝 완료, 저장:', INST_CKPT)

In [ ]:
@torch.no_grad()
def chat(prompt, max_new_tokens=150, temperature=0.7, top_k=40, rep_penalty=1.3):
    model.eval()
    ids=sp.encode(f'### 질문: {prompt}\n### 답변:')
    x=torch.tensor([ids],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        xc=x[:,-block_size:]
        with torch.amp.autocast(device_type='cuda',dtype=getattr(torch,dtype)):
            logits,_=model(xc)
        logits=logits[:,-1,:]/temperature
        for t in set(x[0].tolist()): logits[0,t]/=rep_penalty   
        v,_=torch.topk(logits, min(top_k,logits.size(-1))); logits[logits<v[:,[-1]]]=-float('inf')
        nxt=torch.multinomial(F.softmax(logits,dim=-1),1)
        if nxt.item()==sp.eos_id(): break
        x=torch.cat((x,nxt),dim=1)
    out=sp.decode(x[0].tolist())
    return out.split('### 답변:')[-1].strip()
for q in ['인공지능이 뭐야?','건강하게 사는 방법 알려줘','한국의 수도는 어디야?']:
    print('Q:',q); print('A:',chat(q)); print()